[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/4-Ek_Birim_Cember_ve_Telenakil.ipynb)

# Ek Bölüm: Birim Çember ve Kuantum Telenakil

**Qiskit ile Kuantum Programlamaya Giriş** · Qiskit Fall Fest 2026 · Sabancı Üniversitesi · *Zamanımız kalırsa bu bölümü birlikte işleyeceğiz.*

Bu bölüm iki kısımdan oluşuyor:

1. **Birim çember:** Gerçel genlikli bir kubiti düzlemde çizeceğiz ve RY kapısıyla döndüreceğiz.
2. **Kuantum telenakil** (*quantum teleportation*): Yoğun kodlamada Ediz bir kubit göndererek iki klasik bit iletti. Telenakilde bunun tersini yapacağız: Ediz iki klasik bit göndererek bir **kubit durumunu** Filiz'e iletecek.

*Bu bölüm QWorld'ün QBook101 kitabındaki "Visualization of a (Real-Valued) Qubit", "Rotations" ve "Quantum Teleportation" not defterlerinden yararlanır.*

In [ ]:
# Google Colab'da bu hücreyi bir kez çalıştırın (yaklaşık 1 dakika sürer).
# Kendi bilgisayarınızda requirements.txt ile kurulum yaptıysanız bu hücreyi atlayabilirsiniz.
%pip install -q "qiskit[visualization]==2.6.0" "qiskit-aer==0.18.0"

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

simulator = AerSimulator()

## 1. Kubiti çizmek: birim çember

Genlikleri gerçel sayı olan bir kubit durumu $\begin{pmatrix} \alpha \\ \beta \end{pmatrix}$ düzlemde bir noktadır. Vektörün $\ell_2$ normu $1$ olduğundan ($\alpha^2 + \beta^2 = 1$) bu nokta yarıçapı $1$ olan çemberin, yani **birim çemberin** (*unit circle*) üzerindedir. Yatay eksen $|0\rangle$'ın, dikey eksen $|1\rangle$'in genliğidir.

Durumu, yatay eksenle yaptığı $\theta$ açısıyla da yazabiliriz:

$$ |\psi\rangle = \cos\theta\,|0\rangle + \sin\theta\,|1\rangle. $$

Ölçümde $0$'ı $\cos^2\theta$, $1$'i $\sin^2\theta$ olasılıkla görürüz.

Aşağıdaki hücre, durumları birim çember üzerinde çizen bir yardımcı fonksiyon tanımlar. Hücreyi çalıştırmanız yeterli; içeriğini incelemeniz gerekmiyor.

In [ ]:
# @title Yardımcı fonksiyon: birim çember çizimi (çalıştırın)
import matplotlib.pyplot as plt

def draw_unit_circle(states, labels=None):
    """Gerçel genlikli kubit durumlarını birim çember üzerinde ok olarak çizer."""
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), color="lightgray")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.text(1.05, 0.03, "|0⟩", fontsize=11, color="gray")
    ax.text(0.03, 1.05, "|1⟩", fontsize=11, color="gray")
    if labels is None:
        labels = [""] * len(states)
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:brown"]
    for i, (state, label) in enumerate(zip(states, labels)):
        x, y = np.real(np.asarray(state, dtype=complex))
        color = colors[i % len(colors)]
        ax.annotate("", xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", lw=2, color=color))
        ax.text(1.15 * x, 1.15 * y, label, fontsize=12, color=color, ha="center", va="center")
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

In [ ]:
s = 1 / np.sqrt(2)
draw_unit_circle([[1, 0], [0, 1], [s, s], [s, -s]],
                 ["|0⟩", "|1⟩", "|+⟩", "|−⟩"])

Bölüm 1'de gördüğümüz gibi $|0\rangle$ ile $|1\rangle$, $|+\rangle$ ile de $|-\rangle$ birbirine diktir; çizimde de aralarında $90°$ açı var.

### RY kapısı: kubiti döndürmek

RY kapısı, durumu birim çember üzerinde saat yönünün tersine döndürür; bu işleme **döndürme** (*rotation*) denir. Dikkat: Durumu $\theta$ kadar döndürmek için RY kapısına açının **iki katını** veririz: `qc.ry(2 * theta, 0)`. (Bunun nedeni şudur: Kubitin tam gösterimi, **Bloch küresi** (*Bloch sphere*) adı verilen üç boyutlu bir modeldir ve bu modelde açılar iki katına çıkar.)

> **Doğrusal cebir: döndürme matrisi**
>
> Düzlemde $\theta$ kadar döndürme şu matrisle yapılır:
>
> $$ R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} $$
>
> `qc.ry(2 * theta, 0)` kapısının matrisi tam olarak budur. Döndürme vektörün uzunluğunu değiştirmez; bu yüzden birimcil bir matristir.

Örnek: $\theta = 60°$ (yani $\pi/3$) için durum $\cos 60°\,|0\rangle + \sin 60°\,|1\rangle$ olur. Ölçümde $0$'ı $\cos^2 60° = 0{,}25$, $1$'i $\sin^2 60° = 0{,}75$ olasılıkla görmeyi bekleriz.

In [ ]:
theta = np.pi / 3   # 60 derece

qc = QuantumCircuit(1)
qc.ry(2 * theta, 0)
draw_unit_circle([Statevector(qc)], ["|ψ⟩"])

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

### Görev 1

H kapısı kullanmadan, yalnızca RY kapısıyla $|+\rangle$ durumunu hazırlayın. Hangi $\theta$ açısı gerekir?

In [ ]:
theta = 0   # bu değeri değiştirin (açı radyan cinsinden; np.pi kullanabilirsiniz)

qc = QuantumCircuit(1)
qc.ry(2 * theta, 0)

display(Statevector(qc).draw("latex"))
draw_unit_circle([Statevector(qc)], ["|ψ⟩"])

In [ ]:
# kontrol
if Statevector(qc).equiv(Statevector.from_label("+")):
    print("DOĞRU: Bu durum |+⟩ ile aynı.")
else:
    print("Henüz değil. İpucu: |+⟩ durumu yatay eksenle kaç derecelik açı yapar?")

## 2. Kuantum telenakil

### Problem

Ediz'in elinde bir kubit var ve bu kubit Ediz'in bilmediği bir $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ durumunda. Ediz bu durumu Filiz'e iletmek istiyor, ama aralarında yalnızca klasik bir iletişim kanalı var.

Neden zor?

- Ediz kubiti ölçerse yalnızca $0$ ya da $1$ görür; $\alpha$ ve $\beta$ kaybolur.
- Durumu kopyalayıp kopyaları ölçmek de mümkün değil: bilinmeyen bir kuantum durumu kopyalanamaz. Buna **kopyalanamazlık teoremi** (*no-cloning theorem*) denir.

Çözüm: Ediz ile Filiz önceden dolaşık bir çift paylaşır.

### Protokol

Üç kubit kullanıyoruz:

- $q_0$: Ediz'in göndermek istediği $|\psi\rangle$ durumu,
- $q_1$: dolaşık çiftin Ediz'deki yarısı,
- $q_2$: dolaşık çiftin Filiz'deki yarısı.

**Adım 1 — Dolaşık çift.** $q_1$ ve $q_2$ dolaşık hâle getirilir: $q_1$'e H, sonra kontrolü $q_1$, hedefi $q_2$ olan CNOT.

**Adım 2 — Ediz.** Önce kontrolü $q_0$, hedefi $q_1$ olan bir CNOT, sonra $q_0$'a H kapısı uygular. Sonra $q_0$ ve $q_1$'i ölçer. Sonuçlara $m_0$ ve $m_1$ diyelim.

**Adım 3 — Klasik iletişim.** Ediz, $m_0$ ve $m_1$ bitlerini Filiz'e gönderir.

**Adım 4 — Filiz.** Ediz'in ölçümünden sonra Filiz'in kubiti dört durumdan birindedir. Her birinin olasılığı $\frac14$'dir:

- $m_1 = 0$, $m_0 = 0$: $\;\alpha|0\rangle + \beta|1\rangle$ → hiçbir şey yapmaya gerek yok,
- $m_1 = 1$, $m_0 = 0$: $\;\alpha|1\rangle + \beta|0\rangle$ → X uygular,
- $m_1 = 0$, $m_0 = 1$: $\;\alpha|0\rangle - \beta|1\rangle$ → Z uygular,
- $m_1 = 1$, $m_0 = 1$: $\;\alpha|1\rangle - \beta|0\rangle$ → önce X, sonra Z uygular.

Yani Filiz, $m_1 = 1$ ise X, $m_0 = 1$ ise Z uygular. Sonunda Filiz'in kubiti tam olarak $|\psi\rangle$ olur. (Bu dört durumun hesabı QBook101'deki "Quantum Teleportation" not defterinde adım adım yapılmıştır.)

#### Ölçüm sonucuna bağlı kapılar

Filiz'in kapıları Ediz'in ölçüm sonuçlarına bağlıdır. Qiskit'te bunu `if_test` ile yazarız:

```python
with qc.if_test((1, 1)):   # 1 numaralı klasik bit 1 ise
    qc.x(2)                # 2 numaralı kubite X uygula
```

Bloğun içindeki kapılar yalnızca belirtilen klasik bitin değeri istenen değere eşitse uygulanır.

### Devre

Göndereceğimiz durumu birinci kısımdaki gibi RY kapısıyla hazırlıyoruz: $|\psi\rangle = \cos\theta\,|0\rangle + \sin\theta\,|1\rangle$. Filiz kubitini ölçtüğünde $1$'i $\sin^2\theta$ olasılıkla görmeli.

### Görev 2

Ediz'in ve Filiz'in yapmaları gerekenleri ekleyin. Sonra bir sonraki hücreyi çalıştırıp Filiz'in ölçüm olasılıklarını kontrol edin.

In [ ]:
theta = np.pi / 5   # gönderilecek durumun açısı; değiştirebilirsiniz, ama 45 dereceye (π/4) yakın bir açı seçmeyin:
                    # o zaman 1 olasılığı her durumda 1/2 olur ve aşağıdaki test eksik devreyi de DOĞRU sayar

qc = QuantumCircuit(3, 3)

# gönderilecek durum q0'da: |ψ⟩ = cos θ |0⟩ + sin θ |1⟩
qc.ry(2 * theta, 0)
qc.barrier()

# Adım 1: dolaşık çift (q1 Ediz'de, q2 Filiz'de)
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# Adım 2: Ediz. Kodunuzu buraya yazın (CNOT ve H)

qc.measure(0, 0)   # m0
qc.measure(1, 1)   # m1
qc.barrier()

# Adım 3: Ediz, m0 ve m1 bitlerini Filiz'e gönderir (simülasyonda ayrıca kod yazmaya gerek yok)

# Adım 4: Filiz. Kodunuzu buraya yazın (if_test ile X ve Z)

qc.draw("mpl")

In [ ]:
# kontrol: Filiz'in kubitinde 1 görme olasılığı sin²θ olmalı
shots = 4000
measured = qc.copy()     # devrenin bir kopyası
measured.measure(2, 2)   # Filiz kubitini ölçer
counts = simulator.run(measured, shots=shots).result().get_counts()

# sonuç dizileri c2 c1 c0 sırasındadır; Filiz'in sonucu en soldaki karakter
filiz_ones = sum(n for bits, n in counts.items() if bits[0] == "1")
p1 = filiz_ones / shots
expected = np.sin(theta) ** 2

print("Filiz'in kubitinde 1 olasılığı (ölçülen):", round(p1, 3))
print("beklenen (sin²θ):                         ", round(expected, 3))
if abs(p1 - expected) < 0.04:
    print("DOĞRU görünüyor.")
else:
    print("Henüz değil. Ediz'in kapılarını ve Filiz'in düzeltmelerini kontrol edin.")

### Olasılıklar yetmez

Ölçüm olasılıklarının beklenen değerlerle uyuşması, Filiz'in kubitinin tam olarak $|\psi\rangle$ olduğunu kanıtlamaz. Örneğin $\cos\theta\,|0\rangle - \sin\theta\,|1\rangle$ durumu da aynı olasılıkları verir. Daha güçlü bir test yapabiliriz: $|\psi\rangle$'yi hazırlarken kubiti $\theta$ kadar döndürmüştük. Filiz'in kubitini $\theta$ kadar **geri** döndürürsek, kubit tam olarak $|\psi\rangle$ durumundaysa $|0\rangle$'a döner ve ölçümde **her zaman** $0$ görürüz.

### Ek görev: geri alma testi

Görev 2'deki devrenin bir kopyasını alıyoruz. Filiz'in kubitini ölçmeden önce $\theta$ kadar geri döndürün: `test.ry(-2 * theta, 2)`. Filiz'in sonucu her tekrarda $0$ çıkıyor mu?

In [ ]:
test = qc.copy()   # Görev 2'deki devrenin bir kopyası

# kodunuzu buraya yazın: Filiz'in kubitini (q2) θ kadar geri döndürün

test.measure(2, 2)
counts = simulator.run(test, shots=1000).result().get_counts()
filiz_ones = sum(n for bits, n in counts.items() if bits[0] == "1")
print("Filiz'in 1 gördüğü tekrar sayısı:", filiz_ones, "/ 1000")

### Sonuç ve notlar

- Ediz yalnızca iki klasik bit gönderdi; Filiz'in kubiti $|\psi\rangle$ durumuna geldi.
- Ediz'in kubitindeki durum ölçümle bozuldu. Yani durum kopyalanmadı, **taşındı**.
- Telenakil ışıktan hızlı iletişim sağlamaz: Filiz, Ediz'in iki bitini almadan düzeltmeyi yapamaz. Düzeltmeden önce Filiz'in kubiti tamamen rastgele sonuçlar verir.
- Yoğun kodlama ile telenakil birbirinin tersidir: biri bir kubitle iki bit, diğeri iki bitle bir kubit iletir. İkisi de önceden paylaşılmış bir dolaşık çift kullanır.

### Daha fazlası

- Qiskit: [ibm.com/quantum/qiskit](https://www.ibm.com/quantum/qiskit)
- QWorld QBook101, "Quantum Teleportation": [qworld.net/qbook101](https://qworld.net/qbook101/)
- QTurkey (Kuantum Türkiye) etkinlikleri: [qturkey.org/events](https://qturkey.org/events/)
- IBM Quantum Learning, *Basics of Quantum Information*: [quantum.cloud.ibm.com/learning](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information)